# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described via a Croissant schema available at:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This lets you inspect both the structure and the contents of the data.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's inspect available record sets, and display their IDs and fields, referencing all entities via their Croissant `@id` property.

We will also show an example record from each record set.

In [ ]:
# List available record sets with their @id
record_sets = list(dataset.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name')})")

# Display field @ids for each record set
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("Fields:@ids")
        for f in fields:
            print(f"  - {f['@id']} (name: {f.get('name')})")
    else:
        print("  (No fields defined)")

    # Display the first example record (referencing all fields by @id)
    try:
        example = next(dataset.records(record_set=rs['@id']))
        print("Sample record (by @id):")
        for key in example.keys():
            print(f"  - {key}: {example[key]}")
    except StopIteration:
        print("No records present.")

## 3. Data Extraction

Load entire record sets into Pandas DataFrames for analysis. All references use Croissant `@id`s.

We'll load each available record set by its `@id`.

In [ ]:
dataframes = {}
all_recordset_ids = [rs['@id'] for rs in dataset.record_sets]

for recordset_id in all_recordset_ids:
    records = list(dataset.records(record_set=recordset_id))
    df = pd.DataFrame(records)
    dataframes[recordset_id] = df
    print(f"Loaded record set {recordset_id} with {df.shape[0]} records and columns:")
    print(list(df.columns))

# For demonstration, select the first record set for further analysis
if len(all_recordset_ids) > 0:
    selected_record_set_id = all_recordset_ids[0]
    print(f"\nPreview records from {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)

Let's process the main dataset (the first record set) to demonstrate:
- Filtering on a numeric field (by `@id`)
- Normalizing a numeric field
- Grouping data by a categorical field

All columns will be referenced using Croissant `@id`.

In [ ]:
# --- Identify numeric and group fields by their Croissant @id ---

main_df = dataframes[selected_record_set_id]

# Automatically detect a numeric field (by @id) for demonstration
numeric_field_id = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col  # Take the first numeric column
        break

if numeric_field_id is None:
    print("No numeric field found. Please inspect the DataFrame to select a field.")
else:
    print(f"Using numeric field: {numeric_field_id}")

    # Set a threshold for filtering
    threshold = main_df[numeric_field_id].mean()  # Use mean as demo threshold

    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical field (not the index or the numeric field)
    group_field_id = None
    for col in main_df.columns:
        if col != numeric_field_id and main_df[col].dtype == 'object':
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No suitable group field (categorical by @id) found. Inspect columns:", list(main_df.columns))


## 5. Visualization

Visualize data distributions or relationships using the most informative fields. We will use column `@id`s for all axes/labels.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If grouped_df exists, show barplot
if 'grouped_df' in locals() and group_field_id:
    plt.figure(figsize=(10,5))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a FAIR-compliant biomedical dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all schema entities by their Croissant `@id`. You can adapt this workflow to analyze other fields, run statistical tests, or build machine learning models on top of this harmonized structure.

Key steps:
- All schema entities are referenced using Croissant `@id` for traceable and interoperable analytics.
- Metadata and tabular data are programmatically loaded and transformed for EDA.

**Try adapting this notebook for your custom filter/group/select needs, always using `@id` referencing for FAIR data practices!**